<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama ollama tavily-python

## Without web search

In [1]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="qwen3.5:cloud",
    model_provider="ollama",
    base_url="https://ollama.com",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [3]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=llm)

messages = [HumanMessage(content="How up to date is your training knowledge?")]

start_time = time.time()
response = agent.invoke(input={"messages": messages})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 3.53s seconds
================================ Human Message =================================

How up to date is your training knowledge?
================================== Ai Message ==================================

My training data is current up to **2026**. This means my knowledge and capabilities are based on information available until that date. I may not have awareness of events, data, or developments that occurred after this cutoff. For the most accurate and timely information, especially on rapidly evolving topics, I recommend cross-checking with up-to-date sources. How can I assist you within this scope? 😊


## Add web search tool

In [4]:
from tavily import TavilyClient
from typing import Dict, Any
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, domains: list) -> Dict[str, Any]:
    """
    Search the web for information
    """

    return tavily_client.search(
        query=query,
        include_domains=domains
    )

web_search.invoke(input={
"query": "List of public holidays in Malaysia for 2026.",
"domains": ["timeanddate.com", "officeholidays.com"]
})

{'query': 'List of public holidays in Malaysia for 2026.',
 'response_time': 0.95,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.timeanddate.com/holidays/malaysia/2026',
   'title': 'Holidays and Observances in Malaysia in 2026 - Time and Date',
   'content': "| Jan 1 | Thursday | New Year's Day | State Holiday | All except JHR, KDH, KTN, PLS, TRG |. | Apr 26 | Sunday | Birthday of the Sultan of Terengganu | State Holiday | Terengganu |. | May 26 | Tuesday | Day of Arafat (Tentative Date) | State Holiday | Kelantan, Terengganu |. | May 28 | Thursday | Hari Raya Haji (Day 2) (Tentative Date) | State Holiday | Kedah, Perlis |. | May 28 | Thursday | Hari Raya Haji (Day 2) (Tentative Date) | Federal Public Holiday | Kelantan, Terengganu |. | May 31 | Sunday | Second Day of Harvest Festival | State Holiday | Labuan, Sabah |. | Jun 1 | Monday | Second Day of Harvest Festival observed | State Holiday | Labuan, Sabah |. | Jun 21 | Sunday | Sult

In [5]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a helpful assistant.
When using the web_search tool, you MUST pass:
- query: string
- domains: list of domains

Please keep the below structure.
Country code: The country code of the holiday.
Date: The date of the holiday.
Name: The name of the holiday.
"""

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

messages = [HumanMessage(content="""
List of public holidays in Malaysia for 2026.
Only use these domains:
- timeanddate.com,
- officeholidays.com
""")]

start_time = time.time()
response = agent.invoke(input={"messages": messages})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 16.78s seconds
================================ Human Message =================================


List of public holidays in Malaysia for 2026.
Only use these domains:
- timeanddate.com,
- officeholidays.com

================================== Ai Message ==================================
Tool Calls:
  web_search (3cb9c0ee-6e33-4222-aaad-37d2d1c4b8cc)
 Call ID: 3cb9c0ee-6e33-4222-aaad-37d2d1c4b8cc
  Args:
    query: Malaysia public holidays 2026
    domains: ['timeanddate.com', 'officeholidays.com']
================================= Tool Message =================================
Name: web_search

{"query": "Malaysia public holidays 2026", "response_time": 0.91, "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.timeanddate.com/holidays/malaysia/2026", "title": "Holidays and Observances in Malaysia in 2026 - Time and Date", "content": "| Jan 1 | Thursday | New Year's Day | State Holiday | All except JHR, KDH, KTN, PLS, TRG |. | Apr 26

In [6]:
import os
from ollama import Client
from typing import Dict, Any
from langchain.tools import tool

client = Client(
    host="https://ollama.com/api/web_search",
    headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
)

@tool
def web_search(query: str, domains: list) -> Dict[str, Any]:
    """
    Search the web for information
    """

    domains = " OR ".join([f"site:{d}" for d in domains])
    query = f"{query} ({domains})"
    return client.web_search(query=query)

web_search.invoke(input={
"query": "List of public holidays in Malaysia for 2026.",
"domains": ["timeanddate.com", "officeholidays.com"]
})

WebSearchResponse(results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 200020012002200320042005200620072008200920102011201220132014201520162017201820192020202120222023202420252026Today20272028202920302031203220332034203520362037203820392040 Select AllClear AllReset to Default Federal/National Holidays (15)Common Local Holidays (14)Local holidays (28)Important observances (4)Seasons (4)Major Hinduism (0)\n\n## Holidays and Observances in Malaysia in 2026\n\n| Date | Name | Type | Details |\n| --- | --- | --- | --- |\n| Jan 1 | Thursday | New Year's Day | State Holiday | All except JHR, KDH, KTN, PLS, TRG |\n| Jan 14 | Wednesday | Birthday of Yang di-Pertuan Besar | State Holiday | NSN |\n| Jan 17 | Saturday | I

In [7]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a helpful assistant.
When using the web_search tool, you MUST pass:
- query: string
- domains: list of domains

Please keep the below structure.
Country code: The country code of the holiday.
Date: The date of the holiday.
Name: The name of the holiday.
"""

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

messages = [HumanMessage(content="""
List of public holidays in Malaysia for 2026.
Only use these domains:
- timeanddate.com,
- officeholidays.com
""")]

start_time = time.time()
response = agent.invoke(input={"messages": messages})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 21.02s seconds
================================ Human Message =================================


List of public holidays in Malaysia for 2026.
Only use these domains:
- timeanddate.com,
- officeholidays.com

================================== Ai Message ==================================
Tool Calls:
  web_search (df8accad-7804-4ff9-b0d7-7ba11ee84200)
 Call ID: df8accad-7804-4ff9-b0d7-7ba11ee84200
  Args:
    query: Malaysia public holidays 2026
    domains: ['timeanddate.com', 'officeholidays.com']
================================= Tool Message =================================
Name: web_search

results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 20002001200220032004200520062007200820092010201120